In [ ]:
# Kaggle runner: control-variate weak-error benchmark (g1-g4, regimes A-E).
#
# CONVENTION NOTE.  Every other notebook in notebooks/kaggle/ is standalone:
# it inlines its kernels so it runs with Internet OFF and no pushed branch
# (see notebooks/kaggle/README.md).  This one deliberately does the opposite
# and clones the repo, so the experiment code has a single home and cannot
# drift from experiments/run_weak_error.py.  The cost is that you must:
#
#     Settings -> Internet -> On          (requires a phone-verified account)
#
# Accelerator is not needed: this experiment is NumPy/CPU.  Set it to None.
#
# RESUMING ACROSS SESSIONS.  /kaggle/working is wiped between sessions, so to
# continue an interrupted run: "Add Input" -> Notebook Output -> pick this
# notebook's previous version.  The cell copies any weak_error_partial.csv it
# finds under /kaggle/input into the working directory and passes --resume.
# Levels are independently seeded, so a resumed run reproduces an
# uninterrupted one bit for bit.
#
# RUNNING IT IN THE BACKGROUND.  Prefer "Save Version -> Save & Run All
# (Commit)" over an interactive session: the full A-E run takes roughly 5-8 h
# and a batch run is detached from the browser, so a dropped tab cannot kill
# it.  The committed version's output is also exactly what you attach as an
# input to resume a run that hit the session cap.

import json
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

# ============================ EDIT THIS BLOCK =============================
# Kaggle has no interface for setting environment variables, so the run is
# configured here.  Each value may still be overridden by the matching
# WEAK_CV_* environment variable when driving the script from outside Kaggle.

# "smoke" ~6 min, one regime, integrity only.
# "pilot" ~15 min, all schemes and regimes at 20k paths -- the precision gate
#         to run before committing a session to "full".
# "full"  the production run; see the cost note on REGIMES below.
RUN_MODE = "full"

# Branch, tag, or commit SHA.  For a run whose numbers go into the thesis,
# replace this with the SHA printed below so the artefact is pinned to an
# immutable commit; the SHA is recorded in weak_error_run_config.json either
# way.
REF = "main"

# MEASURED COST: about 3 h for the fixed-step schemes plus about 4 h for the
# adaptive ones on a desktop CPU, so the whole A-E run is ~7 h there and can
# exceed Kaggle's 12 h cap on slower hardware.  SPLIT IT: run "A B C" in one
# session, then "D E" in a second with the first session's output attached as
# an input.  None keeps the preset for the chosen RUN_MODE.
REGIMES = None          # "A B C D E"
N_PATHS = None          # "200000"
N_STEPS = None          # "8 16 32 64 128 256 512 1024 2048 4096"
MAX_ADAPTIVE = None     # "1024"   nominal-step cap for KLM / adaptive KL
SCHEMES = None          # subset of "FTE HH ProjEuler KL IF KLM BLT"

# Kaggle CPU sessions stop at 12 h; leave an hour to write outputs.
TIME_BUDGET_S = "39600"

USE_CONTROL_VARIATE = True   # False reproduces the old direct estimator
REPO_URL = "https://github.com/lukemurray01/CIR_MSc_2025-26.git"
# ==========================================================================

# Absolute: run_weak_error.py is launched with cwd=REPO, so a relative
# --out-dir would land inside the clone and be lost with it.
WORK = Path("/kaggle/working").resolve()
REPO = WORK / "cir_repo"
WORK.mkdir(parents=True, exist_ok=True)

def setting(name, value):
    """Editable constant above, overridable by WEAK_CV_<name> if present."""
    override = os.environ.get(f"WEAK_CV_{name}")
    return override if override not in (None, "") else value


RUN_MODE = setting("RUN_MODE", RUN_MODE).strip().lower()

PRESETS = {
    # Integrity check: a few minutes, one boundary regime.
    "smoke": dict(
        regimes="E",
        n_paths="5000",
        n_steps="8 16 32 64 128",
        max_adaptive="128",
    ),
    # Precision gate: every scheme in every regime at a tenth of the
    # production paths, on a ladder deep enough to show whether the variance
    # reductions hold everywhere.  Run this BEFORE committing a session to
    # the full run and check `variance_reduction` and `error_to_se_cv` per
    # series; if a series is unresolved here it will likely stay unresolved
    # at 200,000 paths, since the s.e. only falls by sqrt(10).
    "pilot": dict(
        regimes="A B C D E",
        n_paths="20000",
        n_steps="8 16 32 64 128 256 512 1024",
        max_adaptive="256",
    ),
    # Production: the deep ladder h = 2^-3 .. 2^-12 for the fixed-step
    # schemes.  The adaptive schemes stop at 2^-10: their accepted-step
    # counts run ~30x the nominal level in regime E, and they are NOT put on
    # a shared fine grid (quantising an adaptive step to a 2^-k grid forces
    # every step to one grid step once h_max reaches the spacing, silently
    # collapsing the scheme to a uniform mesh).
    "full": dict(
        regimes="A B C D E",
        n_paths="200000",
        n_steps="8 16 32 64 128 256 512 1024 2048 4096",
        max_adaptive="1024",
    ),
}
if RUN_MODE not in PRESETS:
    raise SystemExit(f"WEAK_CV_RUN_MODE must be one of {sorted(PRESETS)}")
cfg = dict(PRESETS[RUN_MODE])

cfg["regimes"] = setting("REGIMES", REGIMES) or cfg["regimes"]
cfg["n_paths"] = setting("N_PATHS", N_PATHS) or cfg["n_paths"]
cfg["n_steps"] = setting("N_STEPS", N_STEPS) or cfg["n_steps"]
cfg["max_adaptive"] = setting("MAX_ADAPTIVE", MAX_ADAPTIVE) or cfg["max_adaptive"]

REPO_URL = setting("REPO", REPO_URL)
REF = setting("REF", REF)
SCHEMES = (setting("SCHEMES", SCHEMES) or "").strip()
TIME_BUDGET = (setting("TIME_BUDGET_S", TIME_BUDGET_S) or "").strip()

# ---------------------------------------------------------------- resume ---
# Seed the working directory from a previous run attached as an input.
resumed_from = None
for candidate in sorted(Path("/kaggle/input").glob("**/weak_error_partial.csv")):
    shutil.copy(candidate, WORK / "weak_error_partial.csv")
    resumed_from = str(candidate)
    break

resume = (WORK / "weak_error_partial.csv").exists()
print(f"run mode      : {RUN_MODE}")
print(f"resume        : {resume}" + (f" (from {resumed_from})" if resumed_from else ""))

# ----------------------------------------------------------------- clone ---
def _force_remove(func, path, _exc):
    """git keeps its objects read-only; clear the bit and retry."""
    os.chmod(path, 0o700)
    func(path)


if REPO.exists():
    shutil.rmtree(REPO, onerror=_force_remove)
subprocess.run(
    ["git", "clone", "--quiet", REPO_URL, str(REPO)], check=True
)
subprocess.run(["git", "checkout", "--quiet", REF], cwd=REPO, check=True)
commit = subprocess.run(
    ["git", "rev-parse", "HEAD"], cwd=REPO, check=True,
    capture_output=True, text=True,
).stdout.strip()
print(f"repo          : {REPO_URL}")
print(f"ref / commit  : {REF} / {commit}")
if REF in ("main", "master"):
    print(f"  NOTE: to pin this run for the thesis, set REF = \"{commit}\"")

# ------------------------------------------------------------------- run ---
cmd = [
    sys.executable, "experiments/run_weak_error.py",
    "--regimes", *cfg["regimes"].split(),
    "--n-paths", cfg["n_paths"],
    "--n-steps", *cfg["n_steps"].split(),
    "--max-adaptive-steps", cfg["max_adaptive"],
    "--out-dir", str(WORK),
]
if SCHEMES:
    cmd += ["--schemes", *SCHEMES.split()]
if TIME_BUDGET:
    cmd += ["--time-budget-s", TIME_BUDGET]
if resume:
    cmd += ["--resume"]
if not USE_CONTROL_VARIATE or os.environ.get("WEAK_CV_NO_CV", "").strip() == "1":
    cmd += ["--no-control-variate"]

print("command       : " + " ".join(cmd) + "\n", flush=True)

env = dict(os.environ, PYTHONUNBUFFERED="1", MPLBACKEND="Agg")
completed = subprocess.run(cmd, cwd=REPO, env=env)

# --------------------------------------------------------------- archive ---
# Provenance for the reproducibility ledger: exactly what produced the CSVs.
(WORK / "weak_error_run_config.json").write_text(
    json.dumps(
        {
            "run_mode": RUN_MODE,
            "repo": REPO_URL,
            "ref": REF,
            "commit": commit,
            "command": cmd,
            "resumed": resume,
            "resumed_from": resumed_from,
            "returncode": completed.returncode,
        },
        indent=2,
    ),
    encoding="utf-8",
)

artefacts = [
    p for name in (
        "weak_error.csv", "weak_error_orders.csv", "weak_error_partial.csv",
        "weak_error_run_config.json",
    )
    for p in [WORK / name] if p.exists()
]
artefacts += sorted(WORK.glob("weak_error_regime_*.pdf"))
artefacts += sorted(WORK.glob("weak_error_regime_*.png"))

with zipfile.ZipFile(WORK / "weak_error_results.zip", "w",
                     zipfile.ZIP_DEFLATED) as archive:
    for path in artefacts:
        archive.write(path, path.name)

print("\nartefacts in /kaggle/working:")
for path in artefacts:
    print(f"  {path.name:<42} {path.stat().st_size / 1024:9.1f} KiB")
print("  weak_error_results.zip  <- download this")

if completed.returncode != 0:
    raise SystemExit(f"run_weak_error.py exited {completed.returncode}")
